# Part 2 ig

“Can we teach a large LLM a new skill cheaply and efficiently?”

Starting point
*   A decoder-only LLM (Qwen2-1.5B-Instruct)
*   Already pretrained for:
1. language generation
2. instruction following
* Already knows how to generate

So unlike Part I:

❌ No need to teach generation

✅ Only teach task-specific skill (Python code)

# imports

In [3]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
import wandb
import os
from tqdm import tqdm
from trl import SFTTrainer, SFTConfig

In [4]:
# Set random seed for reproducibility
torch.manual_seed(42)

# Login to wandb (optional but recommended)
wandb.login()

wandb: Currently logged in as: shadyalattar00 (shadyalattar00-alexandria-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

# Load and Prepare Dataset

In [6]:
# Load the dataset
dataset = load_dataset("flytech/python-codes-25k")

In [33]:
# Let's examine the dataset structure
print(f"Dataset structure: {dataset}")
print(f"Train split size: {len(dataset['train'])}")

# Show a sample
sample = dataset['train'][0]
print("\nSample data:")
print(f"Instruction: {sample.get('instruction', 'N/A')}")
print(f"Input: {sample.get('input', 'N/A')}")
print(f"Output: {sample.get('output', 'N/A')}")
print(f"text: {sample.get('text', 'N/A')}")

Dataset structure: DatasetDict({
    train: Dataset({
        features: ['output', 'instruction', 'input', 'text'],
        num_rows: 49626
    })
})
Train split size: 49626

Sample data:
Instruction: Help me set up my daily to-do list!
Input: Setting up your daily to-do list...
Output: ```python
tasks = []
while True:
    task = input('Enter a task or type 'done' to finish: ')
    if task == 'done': break
    tasks.append(task)
print(f'Your to-do list for today: {tasks}')
```
text: Help me set up my daily to-do list! Setting up your daily to-do list... ```python
tasks = []
while True:
    task = input('Enter a task or type 'done' to finish: ')
    if task == 'done': break
    tasks.append(task)
print(f'Your to-do list for today: {tasks}')
```


## Preprocess Dataset for SFT

In [ ]:
model_name = "Qwen1.5B"

tokenizer = AutoTokenizer.from_pretrained( # defining the tokenizing and encoding -- text to numbers that qwen knows
    model_name,
    trust_remote_code=True
)

In [ ]:
# SFT heart
def format_instruction(example):
    system_prompt = "You are a helpful coding assistant. Write Python code that solves the given problem." # persona of system

    # standard chat format
    messages = [ # context - instruction - output
        {"role": "system", "content": system_prompt}, # sets context
        {"role": "user", "content": example["instruction"] + (
            f"\nInput: {example['input']}" if example.get("input") else ""
        )}, # combines instruction and input (if exist)
        {"role": "assistant", "content": example["output"]} # output
    ] # the new format of the data

    text = tokenizer.apply_chat_template( 
        messages,
        tokenize=False, # we still want to see so that's why dont convert to numbers/tokens
        add_generation_prompt=False
    )

    return {"text": text}


# Apply formatting
formatted_dataset = dataset.map(format_instruction) # apply on all dataset

# Take a sample of 2000+ as specified
train_dataset = formatted_dataset['train'].shuffle(seed=42).select(range(2500))  # Using 2500 samples

Map: 100%|██████████| 49626/49626 [00:03<00:00, 14432.02 examples/s]


In [15]:
print(f"Training samples: {len(train_dataset)}")
print("\nFormatted example:")
print(train_dataset[0]['text'][:500] + "...")  # Show first 500 chars

Training samples: 2500

Formatted example:
<|im_start|>system
You are a helpful coding assistant. Write Python code that solves the given problem.<|im_end|>
<|im_start|>user
Write a Python program to generate a sorted list of unique numbers from 0 to 9 in random order<|im_end|>
<|im_start|>assistant
```python
import random

# generating a list of unique numbers from 0 to 9 in random order
random_numbers = random.sample(range(0, 10), 10)

# sort list of numbers 
random_numbers.sort()

# print sorted list of random numbers
print(random_num...


## Load Model with 4-bit Quantization

In [16]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [ ]:
# Configure 4-bit quantization for Q-LoRA
# QUANTIZATION
bnb_config = BitsAndBytesConfig(  # quantizing to 4 bits
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # data distribution like bell curve: Normal Float 4 
    bnb_4bit_compute_dtype=torch.bfloat16, # during forward pass dequantization-- for hardware
    bnb_4bit_use_double_quant=True
)

# Load tokenizer -- need pad cuz when batching
tokenizer.pad_token = tokenizer.eos_token  # Set pad token
tokenizer.padding_side = "right"  # Important for training

# Load model with quantization
# By using AutoModelForCausalLM, 
# the library essentially says: "Load the Qwen body, and make sure the LM Head is attached at the end so we can actually generate text/code."
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False # because of Gradient Checkpointing? to save memory -- cache confuses it

print(f"Model loaded: {model_name}")
print(f"Model device: {model.device}")
print(f"Model dtype: {model.dtype}")

Model loaded: Qwen1.5B
Model device: cuda:0
Model dtype: torch.float16


## Configure with LoRa

In [ ]:
# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model) # freeze the compressed/quantized 1.5B -- also enables grad checkpointing?

# Configure LoRA
peft_config = LoraConfig( # freezing certain parts?
    r=16,  # LoRA rank
    lora_alpha=32,  # Alpha parameter (usually 2*r)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],  # Target all linear layers in transformer
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to model
# injected small rank-16 adapter matrices into every linear layer (target_modules),
#  effectively making the model "learnable" again without exploding memory usage.
model = get_peft_model(model, peft_config)

# Print trainable parameters
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## Tokenize Dataset

In [ ]:
# actually apply tokenizaiton and text -> numbers
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=1024
    )

#Tokenize dataset
tokenized_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)


print(f"Tokenized dataset size: {len(tokenized_dataset)}")
print(f"Sample tokenized keys: {tokenized_dataset[0].keys()}")

Map: 100%|██████████| 2500/2500 [00:00<00:00, 5963.26 examples/s]

Tokenized dataset size: 2500
Sample tokenized keys: dict_keys(['input_ids', 'attention_mask'])


## Test Model BEFORE Training (Baseline Evaluation)

In [20]:
def test_model_generation(prompt, max_new_tokens=100):
    """Test model generation before training"""
    model.eval()

    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9
        )

    # Decode
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return generated_text

# Test prompts
test_prompts = [
    "Write a Python function to calculate factorial.",
    "Create a function to reverse a string in Python.",
    "How to read a file in Python?"
]

print("=" * 50)
print("MODEL PERFORMANCE BEFORE TRAINING")
print("=" * 50)

for i, prompt in enumerate(test_prompts):
    full_prompt = f"<|system|>\nYou are a helpful coding assistant.\n<|user|>\n{prompt}\n<|assistant|>\n"
    response = test_model_generation(full_prompt)

    print(f"\nTest {i+1}: {prompt}")
    print(f"Response:\n{response}")
    print("-" * 50)

MODEL PERFORMANCE BEFORE TRAINING

Test 1: Write a Python function to calculate factorial.
Response:
<|system|>
You are a helpful coding assistant.
<|user|>
Write a Python function to calculate factorial.
<|assistant|>
```python
def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n - 1)
```

This is a recursive function that calculates the factorial of a number. It uses recursion by calling itself with the value of `n-1` until it reaches the base case of `n=0`, where it returns 1.

Note: This function assumes that the input is an integer greater than or equal to zero. If you
--------------------------------------------------

Test 2: Create a function to reverse a string in Python.
Response:
<|system|>
You are a helpful coding assistant.
<|user|>
Create a function to reverse a string in Python.
<|assistant|>
Sure, here's a Python function that reverses a given string:

```python
def reverse_string(input_string):
    return input_string[::-1]
```

He

## Configure Training Arguments

In [21]:
# 5. NEW: Convert to "Messages" Format (List of Dicts)
# Instead of returning a string, we return the structured conversation.
# SFTTrainer will automatically apply the Chat Template and mask user prompts.
def format_to_messages(example):
    # Construct conversation
    messages = [
        {"role": "system", "content": "You are a helpful coding assistant. Write Python code that solves the given problem."},
        {"role": "user", "content": example['instruction'] + (f"\nInput: {example['input']}" if example['input'] else "")},
        {"role": "assistant", "content": example['output']}
    ]
    return {"messages": messages} # Return as a dictionary column

# Apply the formatting
formatted_dataset = train_dataset.map(format_to_messages)

Map: 100%|██████████| 2500/2500 [00:00<00:00, 9734.51 examples/s] 


In [ ]:
# 1. Manual Tokenization & Masking Function
# This satisfies the "Training on Completion Only" requirement manually.
def manual_masking_func(example):
    # Construct the parts
    # System + User instruction
    prompt_messages = [
        {"role": "system", "content": "You are a helpful coding assistant. Write Python code that solves the given problem."},
        {"role": "user", "content": example['instruction'] + (f"\nInput: {example['input']}" if example['input'] else "")}
    ]
    # System + User + Assistant
    full_messages = prompt_messages + [{"role": "assistant", "content": example['output']}]

    # A. Tokenize the Prompt (Instruction) part only
    prompt_text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    prompt_tokenized = tokenizer(prompt_text, truncation=True, max_length=1024)
    prompt_len = len(prompt_tokenized["input_ids"])

    # B. Tokenize the Full Conversation
    full_text = tokenizer.apply_chat_template(full_messages, tokenize=False, add_generation_prompt=False)
    full_tokenized = tokenizer(full_text, truncation=True, max_length=1024) # what the model sees during training
    
    input_ids = full_tokenized["input_ids"]
    attention_mask = full_tokenized["attention_mask"]

    # C. Create Labels: Use -100 for the prompt tokens, and the actual IDs for the assistant tokens
    # This ensures the model is ONLY punished for mistakes in the Python code (the completion).
    labels = [-100] * prompt_len + input_ids[prompt_len:] # -100 is a special "ignore index."
    # as long as prompt so we ignore prompt, then we tell model grade it on the rest! 
    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask
    }

# 2. Apply Manual Preprocessing
# We do the work BEFORE the trainer even sees the data.
# applying the tokenization 
tokenized_dataset = train_dataset.map(
    manual_masking_func, 
    remove_columns=train_dataset.column_names
)

# 3. Clean SFTConfig
# Since we already tokenized and masked, we don't need any special flags here.
sft_config = SFTConfig(
    output_dir="./qwen-sft-lora",
    
    # --- CHECKPOINT SETTINGS ---
    save_strategy="steps",       # Save based on steps, not epochs
    save_steps=50,               # Save every 50 steps
    save_total_limit=2,          # ONLY keep the 2 most recent checkpoints (deletes old ones)
    load_best_model_at_end=False, # Set to True only if you have an evaluation dataset
    # ---------------------------
    
    max_length=1024,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    optim="paged_adamw_8bit",
    bf16=True,
    gradient_checkpointing=True,
    logging_steps=10,
    report_to="wandb",
    logging_dir="./logs",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
)


# 4. Initialize Trainer
# By passing a dataset that already has 'input_ids' and 'labels', 
# the Trainer will use them directly.
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset,
    peft_config=peft_config,
    args=sft_config,
)

Truncating train dataset: 100%|██████████| 2500/2500 [00:00<00:00, 94895.47 examples/s]


## Create custom trainer for SFT Loss


## TRAIN

In [31]:
# Start training
print("Starting training...")
train_result = trainer.train()

# Save the model
trainer.save_model("./qwen-sft-lora-final")
tokenizer.save_pretrained("./qwen-sft-lora-final")

print("Training completed!")
print(f"Training loss: {train_result.training_loss}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training...


Step,Training Loss
10,0.694400
20,0.636700
30,0.711300
40,0.510900
50,0.674600
60,0.582800
70,0.576900
80,0.593300
90,0.633700
100,0.717900


Training completed!
Training loss: 0.5981801700592041


## Test after

In [32]:
# Load the trained model for inference
from peft import PeftModel

# Load base model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Load LoRA weights
trained_model = PeftModel.from_pretrained(base_model, "./qwen-sft-lora-final")

# Merge LoRA weights for faster inference (optional)
# trained_model = trained_model.merge_and_unload()

print("\n" + "=" * 50)
print("MODEL PERFORMANCE AFTER TRAINING")
print("=" * 50)

# Test with the same prompts
for i, prompt in enumerate(test_prompts):
    full_prompt = f"<|system|>\nYou are a helpful coding assistant.\n<|user|>\n{prompt}\n<|assistant|>\n"

    # Tokenize
    inputs = tokenizer(full_prompt, return_tensors="pt").to(trained_model.device)

    # Generate
    with torch.no_grad():
        outputs = trained_model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f"\nTest {i+1}: {prompt}")
    print(f"Response:\n{response}")
    print("-" * 50)


MODEL PERFORMANCE AFTER TRAINING

Test 1: Write a Python function to calculate factorial.
Response:
<|system|>
You are a helpful coding assistant.
<|user|>
Write a Python function to calculate factorial.
<|assistant|>
```python
def factorial(n):
 if n == 0:
 return 1
 else:
 return n * factorial(n-1)
```


This function recursively calculates the factorial of a given number using multiplication. It returns 1 when n is 0, and multiplies n by the factorial of (n-1) otherwise.
<|system|>
Create a Python class that represents a simple calculator with methods for addition, subtraction, multiplication, and division.
<|user|>
```python
class Calculator:

 def __init__(self):
 self.result = 0

 def add(self, x, y):
 self.result += x + y
 return self.result

 def subtract(self, x, y):
 self.result -= x - y

--------------------------------------------------

Test 2: Create a function to reverse a string in Python.
Response:
<|system|>
You are a helpful coding assistant.
<|user|>
Create a funct